# MAD Stage 2 — Judge Pipeline
### Model: Qwen/Qwen2.5-14B-Instruct-AWQ · Deterministic (temp=0.0) · Anti-bias

**Inputs:** Upload the DB from Stage 1 (MAD v2 run). `judge_verdicts` must be empty (or partially filled — run is idempotent).

**Outputs:** `judge_verdicts` filled → re-export GRPO/SFT/DPO with Brier rewards.

**Run cells top to bottom. Cell 12 is the main judge run — safe to re-run.**

In [ ]:
# CELL 1 — Install dependencies
%%capture
!pip install vllm aiohttp nest_asyncio pydantic
print('Install complete')

In [ ]:
# CELL 2 — GPU check + fix Colab async
import subprocess, nest_asyncio, logging

nest_asyncio.apply()

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'GPU: {gpu}')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('JUDGE')
print('\u2713 Logging configured')
print('\u2713 nest_asyncio applied')

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB
✓ Logging configured
✓ nest_asyncio applied


In [ ]:
# CELL 3 — Upload DB (the completed Stage 1 DB from MAD v2 run)
from google.colab import files
import sqlite3

print('Upload your completed Stage 1 .db file (with agent_outputs filled):')
uploaded = files.upload()

db_files = [f for f in uploaded.keys() if f.endswith('.db')]
if not db_files:
    raise ValueError('No .db file found')

DB_PATH = f'/content/{db_files[0]}'
print(f'\nDB path: {DB_PATH}')

conn = sqlite3.connect(DB_PATH)
for tbl in ['queries', 'claims', 'agent_outputs', 'agent_deltas', 'judge_verdicts']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<20} {n}')

ao = conn.execute('SELECT COUNT(DISTINCT agent_role) FROM agent_outputs').fetchone()[0]
print(f'\n  Distinct agent roles in DB: {ao}')
roles = conn.execute('SELECT DISTINCT agent_role FROM agent_outputs').fetchall()
print(f'  Roles: {[r[0] for r in roles]}')
conn.close()
print('\u2713 DB loaded')

Upload your completed Stage 1 .db file (with agent_outputs filled):


Saving mad_before_phase1_5090_ragfix_01_ (2) (1).db to mad_before_phase1_5090_ragfix_01_ (2) (1) (1).db

DB path: /content/mad_before_phase1_5090_ragfix_01_ (2) (1) (1).db
  queries              50
  claims               414
  agent_outputs        2484
  agent_deltas         1242
  judge_verdicts       414

  Distinct agent roles in DB: 3
  Roles: ['agent_a', 'agent_b', 'agent_c']
✓ DB loaded


In [ ]:
# CELL 4 — Config

JUDGE_MODEL   = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
VLLM_PORT     = 8002
VLLM_URL      = f'http://localhost:{VLLM_PORT}/v1'
SERVED_NAME   = 'judge'
JUDGE_LOG     = '/content/vllm_judge.log'

# Judge is deterministic: temperature=0 for reproducible labels
JUDGE_TEMP      = 0.0
JUDGE_MAX_TOK   = 800
VLLM_TIMEOUT    = 120

# Concurrency: judge is heavier than agents — lower concurrency
JUDGE_CONCURRENCY = 6

# Prompt limits (same as Stage 1)
MAX_CHUNK_CHARS = 800
MAX_CLAIM_CHARS = 300
MAX_PEER_CHARS  = 500   # per debater per round shown to judge

print('\u2713 Config set')
print(f'  Model:       {JUDGE_MODEL}')
print(f'  Temperature: {JUDGE_TEMP} (deterministic)')
print(f'  Concurrency: {JUDGE_CONCURRENCY} claims in parallel')

✓ Config set
  Model:       Qwen/Qwen2.5-14B-Instruct-AWQ
  Temperature: 0.0 (deterministic)
  Concurrency: 6 claims in parallel


In [ ]:
# CELL 5 — Start vLLM judge server
# Same model, separate port (8002) from Stage 1 agent server (8001)
# temperature=0 enforced at call level; prefix caching benefits judge
# because every claim shares the same system prompt.
import subprocess, time

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                  JUDGE_MODEL,
    '--quantization',           'awq_marlin',
    '--max-model-len',          '8192',
    '--gpu-memory-utilization', '0.88',
    '--max-num-seqs',           '12',
    '--enable-prefix-caching',
    '--port',                   str(VLLM_PORT),
    '--served-model-name',      SERVED_NAME,
    '--trust-remote-code',
    '--dtype',                  'float16',
    '--generation-config',      'vllm',
]

print('Launching vLLM judge server...')
print('Port:', VLLM_PORT)
judge_log  = open(JUDGE_LOG, 'w')
judge_proc = subprocess.Popen(cmd, stdout=judge_log, stderr=subprocess.STDOUT)
print(f'PID: {judge_proc.pid}')
print(f'Log: {JUDGE_LOG}')
print('\u2192 Run CELL 6 to wait for ready (~3-4 min)')

Launching vLLM judge server...
Port: 8002
PID: 2042
Log: /content/vllm_judge.log
→ Run CELL 6 to wait for ready (~3-4 min)


In [ ]:
# CELL 6 — Wait for judge server ready
import requests

def wait_for_server(url, timeout_secs=360):
    print('Waiting for vLLM judge server', end='', flush=True)
    for i in range(timeout_secs // 5):
        try:
            r = requests.get(f'{url}/models', timeout=3)
            if r.status_code == 200:
                models = r.json().get('data', [])
                print(f'\n\u2713 Ready in {i*5}s! Models: {[m["id"] for m in models]}')
                return True
        except Exception:
            pass
        print('.', end='', flush=True)
        time.sleep(5)
    print('\n\u2717 Server failed to start. Tail log:')
    print(subprocess.run(['tail', '-30', JUDGE_LOG], capture_output=True, text=True).stdout)
    return False

ready = wait_for_server(VLLM_URL)
if ready:
    kv = subprocess.run(['grep', '-i', 'kv cache', JUDGE_LOG], capture_output=True, text=True).stdout.strip()
    if kv: print(f'KV cache: {kv[-200:]}')

Waiting for vLLM judge server...............................
✓ Ready in 155s! Models: ['judge']
KV cache: _cache_utils.py:1708] GPU KV cache size: 129,184 tokens
(EngineCore pid=2357) INFO 05-06 09:01:18 [core.py:299] init engine (profile, create kv cache, warmup model) took 53.14 s (compilation: 37.39 s)


In [ ]:
# CELL 7 — Pydantic schema for judge output
from pydantic import BaseModel, Field
from typing import List, Optional

class JudgeOutput(BaseModel):
    v_label:          float = Field(ge=0.0, le=1.0)    # 1.0=SUPPORTED 0.5=PARTIAL 0.0=NOT_SUPPORTED
    judge_confidence: float = Field(ge=0.1, le=0.95)
    judge_reasoning:  str
    evidence_chunk_ids: List[str] = []

# Validate v_label is one of the expected values
VALID_V_LABELS = {0.0, 0.5, 1.0}

def normalize_v_label(raw) -> Optional[float]:
    """Map judge verdict text or float to 0.0 / 0.5 / 1.0 / None (IDK)."""
    if isinstance(raw, (int, float)):
        v = float(raw)
        if v >= 0.8:  return 1.0
        if v >= 0.3:  return 0.5
        return 0.0
    s = str(raw).upper().strip()
    mapping = {
        'SUPPORTED': 1.0, 'SUPPORT': 1.0, '1.0': 1.0, '1': 1.0,
        'PARTIAL': 0.5,  'PARTIALLY SUPPORTED': 0.5, '0.5': 0.5,
        'NOT_SUPPORTED': 0.0, 'NOT SUPPORTED': 0.0, 'NOTSUPPORTED': 0.0,
        'UNSUPPORTED': 0.0, '0.0': 0.0, '0': 0.0,
        'IDK': None, 'UNKNOWN': None, 'INSUFFICIENT': None,
    }
    return mapping.get(s, None)

print('\u2713 Judge schema defined')
print('  v_label mapping: SUPPORTED=1.0, PARTIAL=0.5, NOT_SUPPORTED=0.0, IDK=None (skipped)')

✓ Judge schema defined
  v_label mapping: SUPPORTED=1.0, PARTIAL=0.5, NOT_SUPPORTED=0.0, IDK=None (skipped)


In [ ]:
# CELL 8 — DB helpers for Stage 2
import sqlite3, json, threading, uuid

db_lock = threading.Lock()

# ── Readers ───────────────────────────────────────────────────────────────────
def load_all_queries():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute('SELECT * FROM queries ORDER BY query_id').fetchall()
    conn.close()
    result = []
    for r in rows:
        d = dict(r)
        d['rag_chunks']    = json.loads(d['rag_chunks'])
        d['rag_chunk_ids'] = json.loads(d['rag_chunk_ids'])
        result.append(d)
    return result

def load_claims_for_query(query_id):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        'SELECT * FROM claims WHERE query_id=? ORDER BY claim_index', (query_id,)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def load_agent_outputs_for_claim(claim_id):
    """Returns dict: {(agent_role, round_num): row_dict}"""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        'SELECT * FROM agent_outputs WHERE claim_id=? ORDER BY agent_role, round_num',
        (claim_id,)
    ).fetchall()
    conn.close()
    result = {}
    for r in rows:
        d = dict(r)
        if d.get('evidence_cited'):
            try: d['evidence_cited'] = json.loads(d['evidence_cited'])
            except: d['evidence_cited'] = []
        result[(d['agent_role'], d['round_num'])] = d
    return result

def get_judged_claim_ids():
    """Return set of claim_ids already in judge_verdicts."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute('SELECT claim_id FROM judge_verdicts').fetchall()
    conn.close()
    return {r[0] for r in rows}

# ── Writer ────────────────────────────────────────────────────────────────────
def write_judge_verdict(
    claim_id, v_label, judge_confidence, judge_reasoning,
    evidence_chunk_ids, raw_response, latency_ms
):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO judge_verdicts
            (verdict_id, claim_id, v_label, judge_confidence, judge_reasoning,
             evidence_chunk_ids, judge_model, raw_response, latency_ms, timestamp)
            VALUES (?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP)
        """, (
            str(uuid.uuid4()), claim_id,
            v_label, judge_confidence, judge_reasoning,
            json.dumps(evidence_chunk_ids),
            JUDGE_MODEL, raw_response[:4000], latency_ms
        ))
        conn.commit()
        conn.close()

print('\u2713 DB helpers ready')
print(f'  Queries: {len(load_all_queries())}')
print(f'  Already judged: {len(get_judged_claim_ids())}')

✓ DB helpers ready
  Queries: 50
  Already judged: 0


In [ ]:
# CELL 9 — Judge system prompt
# Anti-bias design:
#   1. Evidence-first: judge forms own view before reading debate
#   2. Anti-majority: explicit warning that consensus != truth
#   3. Deterministic: temp=0.0, consistent labels

JUDGE_SYSTEM = """You are an independent regulatory compliance judge.

YOUR TASK: Assign a ground-truth label (v_label) for whether a CLAIM is supported by RETRIEVED EVIDENCE.

EVALUATION PROCESS — follow this order:
STEP 1: Read the RETRIEVED EVIDENCE carefully. Form your own verdict on the claim.
STEP 2: Read the DEBATE TRANSCRIPT. Update your verdict ONLY if a debater cited specific chunk evidence you missed.
STEP 3: Ignore any debater who made assertions without citing specific chunk text.

CRITICAL RULES:
1. BASE your verdict on the RETRIEVED EVIDENCE, not on debater consensus.
   The fact that 2 or 3 debaters agree does NOT make their verdict correct.
   Majority opinion is NOT evidence. You are the independent arbiter.
2. v_label values:
   - 1.0 = The evidence clearly and directly supports the claim.
   - 0.5 = The evidence supports PART of the claim but not all. Name exactly what part.
   - 0.0 = The evidence contradicts or is completely silent on the claim.
   - null = Genuinely too ambiguous to judge (use rarely — prefer 0.0 when evidence is silent).
3. judge_confidence: your certainty. Range 0.10 to 0.95. Reflects evidence strength, not consensus.
4. evidence_chunk_ids: list the chunk IDs that were most decisive for your verdict.
5. Do NOT repeat debater reasoning verbatim. Write your own independent analysis.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON.
{
    \"v_label\": 1.0,
    \"judge_confidence\": 0.9,
    \"judge_reasoning\": \"Independent analysis citing specific chunk content...\",
    \"evidence_chunk_ids\": [\"chunk_id_1\", \"chunk_id_2\"]
}"""

print('\u2713 Judge system prompt defined')
print(f'  Length: {len(JUDGE_SYSTEM)} chars (~{len(JUDGE_SYSTEM)//4} tokens)')

✓ Judge system prompt defined
  Length: 1585 chars (~396 tokens)


In [ ]:
# CELL 10 — Judge prompt builder
# Randomizes debater order per claim (anti-position-bias).
# Agent identity and confidence_internal are NEVER shown to judge.
import random

def format_chunks(chunks):
    parts = []
    for i, c in enumerate(chunks):
        text = c['text'][:MAX_CHUNK_CHARS]
        if len(c['text']) > MAX_CHUNK_CHARS:
            text += '... [truncated]'
        parts.append(f"[Chunk {i+1} | ID: {c['chunk_id']} | Source: {c['source_file']}]\n{text}")
    return '\n\n'.join(parts)

def format_debater_output(label, r0_row, r1_row):
    """Format one debater's R0 + R1 for the judge. No confidence_internal, no agent_role."""
    def fmt_round(row, round_label):
        ev_list = row.get('evidence_cited', [])
        if isinstance(ev_list, str):
            try: ev_list = json.loads(ev_list)
            except: ev_list = []
        ev_ids = ', '.join(
            e['chunk_id'] if isinstance(e, dict) else str(e)
            for e in ev_list[:3]
        ) or 'none cited'
        reasoning = (row.get('reasoning') or '')[:MAX_PEER_CHARS]
        return (
            f"{round_label}\n"
            f"  Verdict:  {row.get('verdict', 'IDK')}\n"
            f"  Reasoning: {reasoning}\n"
            f"  Evidence cited: {ev_ids}"
        )
    lines = [f'=== {label} ===']
    if r0_row: lines.append(fmt_round(r0_row, 'Round 0 (independent):'))
    if r1_row: lines.append(fmt_round(r1_row, 'Round 1 (after seeing peers):'))
    return '\n'.join(lines)

def build_judge_prompt(query, claim, rag_chunks, agent_outputs_dict):
    """
    agent_outputs_dict: {(agent_role, round_num): row_dict}
    Randomizes debater label assignment per claim (anti-position-bias).
    Seed = claim_id for reproducibility.
    """
    # Get distinct agent roles present in data
    roles = sorted(set(k[0] for k in agent_outputs_dict.keys()))

    # Shuffle roles using claim_id as seed (reproducible)
    rng = random.Random(claim['claim_id'])
    shuffled = roles[:]
    rng.shuffle(shuffled)

    # Build debater sections
    debater_sections = []
    for i, role in enumerate(shuffled, 1):
        label  = f'Debater {i}'
        r0_row = agent_outputs_dict.get((role, 0))
        r1_row = agent_outputs_dict.get((role, 1))
        if r0_row or r1_row:
            debater_sections.append(format_debater_output(label, r0_row, r1_row))

    return (
        f"USER QUERY: {query['user_query']}\n\n"
        f"CLAIM TO VERIFY:\n{claim['claim_text'][:MAX_CLAIM_CHARS]}\n\n"
        f"RETRIEVED EVIDENCE:\n{format_chunks(rag_chunks)}\n\n"
        f"DEBATE TRANSCRIPT:\n"
        + '\n\n'.join(debater_sections)
        + '\n\nBased on the RETRIEVED EVIDENCE (Step 1 first, then debate), provide your verdict.'
    )

# Estimate worst-case prompt size
queries = load_all_queries()
sample_q = max(queries, key=lambda q: len(json.dumps(q['rag_chunks'])))
sample_c = load_claims_for_query(sample_q['query_id'])[0]
sample_ao = load_agent_outputs_for_claim(sample_c['claim_id'])
sample_p = build_judge_prompt(sample_q, sample_c, sample_q['rag_chunks'], sample_ao)
print('\u2713 Judge prompt builder ready')
print(f'  Worst-case prompt: ~{(len(sample_p)+len(JUDGE_SYSTEM))//4} tokens (server limit=8192)')

✓ Judge prompt builder ready
  Worst-case prompt: ~2496 tokens (server limit=8192)


In [ ]:
# CELL 11 — Judge async client
import aiohttp, asyncio, re, json, time

def fix_json_newlines(s: str) -> str:
    """Escape literal newlines inside JSON string values."""
    result, in_string, escape_next = [], False, False
    for ch in s:
        if escape_next:
            result.append(ch); escape_next = False
        elif ch == '\\':
            result.append(ch); escape_next = True
        elif ch == '"':
            result.append(ch); in_string = not in_string
        elif in_string and ch == '\n':
            result.append('\\n')
        elif in_string and ch == '\r':
            result.append('\\r')
        else:
            result.append(ch)
    return ''.join(result)

def parse_judge_json(raw: str):
    """6-strategy JSON parser — same as Stage 1 agents."""
    raw = raw.strip()
    for fn in [
        lambda r: json.loads(r),
        lambda r: json.loads(fix_json_newlines(r)),
        lambda r: json.loads(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1)),
        lambda r: json.loads(fix_json_newlines(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1))),
        lambda r: json.loads(re.search(r'\{.*\}', r, re.DOTALL).group(0)),
        lambda r: json.loads(fix_json_newlines(re.search(r'\{.*\}', r, re.DOTALL).group(0))),
    ]:
        try:
            result = fn(raw)
            if isinstance(result, dict) and 'v_label' in result:
                return result
        except Exception:
            continue
    # Regex fallback
    vm = re.search(r'"v_label"\s*:\s*([0-9.]+|null)', raw)
    cm = re.search(r'"judge_confidence"\s*:\s*([0-9.]+)', raw)
    rm = re.search(r'"judge_reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"', raw)
    return {
        'v_label':          float(vm.group(1)) if vm and vm.group(1) != 'null' else None,
        'judge_confidence': float(cm.group(1)) if cm else 0.5,
        'judge_reasoning':  rm.group(1) if rm else f'[PARSE_FAILED] {raw[:400]}',
        'evidence_chunk_ids': [],
        '_parse_failed': True,
    }

async def call_judge(session, user_prompt: str, claim_id: str):
    """
    Returns: (parsed_dict, raw_text, latency_ms)
    Temperature=0.0 — deterministic.
    One retry on parse failure.
    """
    for attempt in range(2):
        prompt = user_prompt
        if attempt == 1:
            prompt += '\n\nREMINDER: Output ONLY a valid JSON object with keys v_label, judge_confidence, judge_reasoning, evidence_chunk_ids.'

        payload = {
            'model':       SERVED_NAME,
            'messages':    [
                {'role': 'system', 'content': JUDGE_SYSTEM},
                {'role': 'user',   'content': prompt},
            ],
            'temperature': JUDGE_TEMP,
            'max_tokens':  JUDGE_MAX_TOK,
            'stream':      False,
        }

        start = time.time()
        try:
            async with session.post(
                f'{VLLM_URL}/chat/completions',
                json=payload,
                timeout=aiohttp.ClientTimeout(total=VLLM_TIMEOUT),
            ) as resp:
                resp.raise_for_status()
                data       = await resp.json()
                latency_ms = int((time.time()-start)*1000)
                raw_text   = data['choices'][0]['message']['content']
        except Exception as e:
            latency_ms = int((time.time()-start)*1000)
            logger.error(f'Judge call error | {claim_id[:8]} | {e}')
            return ({'v_label': None, 'judge_confidence': 0.0,
                     'judge_reasoning': f'[CALL_ERROR] {e}',
                     'evidence_chunk_ids': [], '_call_failed': True},
                    str(e), latency_ms)

        parsed = parse_judge_json(raw_text)

        if parsed.get('_parse_failed'):
            logger.warning(f'Judge parse fail attempt {attempt+1} | {claim_id[:8]} | {raw_text[:60]}')
            if attempt == 0:
                await asyncio.sleep(2)
                continue

        # Normalize v_label to 0.0 / 0.5 / 1.0 / None
        raw_v = parsed.get('v_label')
        parsed['v_label'] = normalize_v_label(raw_v)

        # Clamp confidence
        try:
            parsed['judge_confidence'] = max(0.10, min(0.95, float(parsed.get('judge_confidence', 0.5))))
        except:
            parsed['judge_confidence'] = 0.5

        return parsed, raw_text, latency_ms

    # Should not reach here
    return ({'v_label': None, 'judge_confidence': 0.0,
             'judge_reasoning': '[EXHAUSTED]', 'evidence_chunk_ids': [],
             '_parse_failed': True}, '[EXHAUSTED]', 0)

print('\u2713 Judge async client ready (temp=0.0, 6-strategy parser, fix_json_newlines)')

✓ Judge async client ready (temp=0.0, 6-strategy parser, fix_json_newlines)


In [ ]:
# CELL 12 — MAIN JUDGE RUN (all claims — idempotent, safe to re-run)
# Skips claims that already have a row in judge_verdicts.
import asyncio, time, json
from collections import Counter

async def judge_all():
    queries      = load_all_queries()
    judged_ids   = get_judged_claim_ids()
    sem          = asyncio.Semaphore(JUDGE_CONCURRENCY)
    start_time   = time.time()

    total_claims   = sum(len(load_claims_for_query(q['query_id'])) for q in queries)
    pending_claims = total_claims - len(judged_ids)

    print(f'Stage 2 Judge — {len(queries)} queries | {total_claims} total claims')
    print(f'Already judged: {len(judged_ids)} | Pending: {pending_claims}')
    print(f'Concurrency: {JUDGE_CONCURRENCY} claims in parallel\n')

    verdict_counts = Counter()
    skip_count = 0
    error_count = 0

    async def judge_one(session, query, claim):
        nonlocal skip_count, error_count
        if claim['claim_id'] in judged_ids:
            skip_count += 1
            return

        async with sem:
            try:
                ao = load_agent_outputs_for_claim(claim['claim_id'])
                if not ao:
                    logger.warning(f'No agent outputs for {claim["claim_id"][:8]} — skipping')
                    return

                prompt = build_judge_prompt(query, claim, query['rag_chunks'], ao)
                parsed, raw, lat = await call_judge(session, prompt, claim['claim_id'])

                v_label = parsed.get('v_label')
                if v_label is None:
                    label_str = 'IDK'
                elif v_label >= 0.9: label_str = 'SUPPORTED'
                elif v_label >= 0.4: label_str = 'PARTIAL'
                else: label_str = 'NOT_SUPPORTED'

                verdict_counts[label_str] += 1

                # Write to DB (even if v_label=None — stores the IDK for review)
                write_judge_verdict(
                    claim_id          = claim['claim_id'],
                    v_label           = v_label,
                    judge_confidence  = parsed.get('judge_confidence', 0.5),
                    judge_reasoning   = parsed.get('judge_reasoning', ''),
                    evidence_chunk_ids= parsed.get('evidence_chunk_ids', []),
                    raw_response      = raw,
                    latency_ms        = lat,
                )

                logger.info(
                    f'{claim["claim_id"][:8]} | {label_str} (v={v_label}) '
                    f'| conf={parsed.get("judge_confidence",0):.2f} | {lat}ms'
                )

            except Exception as e:
                error_count += 1
                logger.error(f'Claim {claim["claim_id"][:8]} failed: {e}')

    async with aiohttp.ClientSession() as session:
        for i, query in enumerate(queries):
            claims  = load_claims_for_query(query['query_id'])
            pending = [c for c in claims if c['claim_id'] not in judged_ids]

            if not pending:
                print(f'[{i+1:02d}/50] {query["query_id"]} \u2014 all {len(claims)} done, skipping')
                continue

            print(f'[{i+1:02d}/50] {query["query_id"]} | {len(pending)}/{len(claims)} pending | {query["user_query"][:55]}...')

            await asyncio.gather(*[judge_one(session, query, c) for c in pending])

            # Refresh judged set after each query
            judged_ids = get_judged_claim_ids()

            elapsed = time.time() - start_time
            done_q  = i + 1
            rate    = done_q / elapsed if elapsed > 0 else 1
            eta_s   = (len(queries) - done_q) / rate
            print(f'    elapsed={elapsed/60:.1f}min | ETA={eta_s/60:.1f}min | verdicts so far: {dict(verdict_counts)}')
            print()

    total = time.time() - start_time
    print(f'\n\u2713 Judge done in {total/60:.1f} minutes')
    print(f'Verdict distribution: {dict(verdict_counts)}')
    print(f'Errors: {error_count} | Skipped (already done): {skip_count}')

asyncio.run(judge_all())

Stage 2 Judge — 50 queries | 414 total claims
Already judged: 0 | Pending: 414
Concurrency: 6 claims in parallel

[01/50] q_001 | 11/11 pending | What is the relationship between self-reported physical...
    elapsed=0.1min | ETA=6.2min | verdicts so far: {'SUPPORTED': 2, 'PARTIAL': 4, 'NOT_SUPPORTED': 5}

[02/50] q_002 | 2/2 pending | What must be disregarded in the proceeding according to...
    elapsed=0.2min | ETA=3.9min | verdicts so far: {'SUPPORTED': 2, 'PARTIAL': 4, 'NOT_SUPPORTED': 7}

[03/50] q_003 | 5/5 pending | What does 'contrary' mean in the context of comparing S...
    elapsed=0.2min | ETA=3.4min | verdicts so far: {'SUPPORTED': 7, 'PARTIAL': 4, 'NOT_SUPPORTED': 7}

[04/50] q_004 | 10/10 pending | What must a covered entity do when making routine and r...
    elapsed=0.3min | ETA=3.6min | verdicts so far: {'SUPPORTED': 9, 'PARTIAL': 4, 'NOT_SUPPORTED': 15}

[05/50] q_005 | 9/9 pending | What is the requirement for using the Employer Identifi...
    elapsed=0.4min | ETA

In [ ]:
# CELL 13 — Verify judge results
import sqlite3

conn = sqlite3.connect(DB_PATH)

print('=== TABLE COUNTS ===')
for tbl in ['queries', 'claims', 'agent_outputs', 'agent_deltas', 'judge_verdicts']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<20} {n}')

print('\n=== JUDGE VERDICT DISTRIBUTION ===')
rows = conn.execute("""
    SELECT
        CASE
            WHEN v_label >= 0.9 THEN 'SUPPORTED (1.0)'
            WHEN v_label >= 0.4 THEN 'PARTIAL (0.5)'
            WHEN v_label IS NULL THEN 'IDK (null)'
            ELSE 'NOT_SUPPORTED (0.0)'
        END as verdict,
        COUNT(*) as n,
        ROUND(AVG(judge_confidence), 3) as avg_conf
    FROM judge_verdicts
    GROUP BY 1 ORDER BY 2 DESC
""").fetchall()
total_j = conn.execute('SELECT COUNT(*) FROM judge_verdicts').fetchone()[0]
for r in rows:
    pct = r[1]*100//total_j if total_j else 0
    print(f'  {r[0]:<22} {r[1]:>4} ({pct}%) | avg_conf={r[2]}')

print('\n=== MISSING JUDGE VERDICTS ===')
missing = conn.execute("""
    SELECT c.query_id, c.claim_id, c.claim_text
    FROM claims c
    WHERE c.claim_id NOT IN (SELECT claim_id FROM judge_verdicts)
""").fetchall()
print(f'  Claims without verdict: {len(missing)}')
for m in missing[:5]:
    print(f'  {m[0]} | {m[1][:12]} | {m[2][:60]}')

print('\n=== JUDGE vs AGENT AGREEMENT (R1) ===')
# Fix: CTEs don't have rowid in SQLite — use MAX(cnt) correlated subquery instead
agree = conn.execute("""
    WITH judge_cat AS (
        SELECT claim_id,
               CASE WHEN v_label>=0.9 THEN 'SUPPORTED'
                    WHEN v_label>=0.4 THEN 'PARTIAL'
                    ELSE 'NOT_SUPPORTED' END as judge_verdict
        FROM judge_verdicts WHERE v_label IS NOT NULL
    ),
    agent_majority AS (
        SELECT claim_id, verdict, COUNT(*) as cnt
        FROM agent_outputs WHERE round_num=1
        GROUP BY claim_id, verdict
    ),
    top_agent AS (
        SELECT am.claim_id, am.verdict as majority_verdict
        FROM agent_majority am
        WHERE am.cnt = (
            SELECT MAX(am2.cnt)
            FROM agent_majority am2
            WHERE am2.claim_id = am.claim_id
        )
        GROUP BY am.claim_id
    )
    SELECT
        SUM(CASE WHEN jc.judge_verdict = ta.majority_verdict THEN 1 ELSE 0 END) as agree,
        COUNT(*) as total
    FROM judge_cat jc
    JOIN top_agent ta USING (claim_id)
""").fetchone()
if agree and agree[1]:
    pct = agree[0]*100//agree[1]
    print(f'  Judge agrees with agent majority: {agree[0]}/{agree[1]} ({pct}%)')
    if pct > 80:
        print(f'  ⚠ High agreement ({pct}%) — judge may be influenced by majority consensus')
    elif pct < 50:
        print(f'  ✓ Low agreement ({pct}%) — judge is making independent decisions')
    else:
        print(f'  ~ Moderate agreement ({pct}%) — judge partially independent')

print('\n=== JUDGE CONFIDENCE DISTRIBUTION ===')
conf_rows = conn.execute("""
    SELECT
        CASE WHEN judge_confidence >= 0.9 THEN '0.90-0.95'
             WHEN judge_confidence >= 0.8 THEN '0.80-0.89'
             WHEN judge_confidence >= 0.7 THEN '0.70-0.79'
             WHEN judge_confidence >= 0.6 THEN '0.60-0.69'
             ELSE '<0.60' END as bucket,
        COUNT(*) as n
    FROM judge_verdicts
    GROUP BY 1 ORDER BY 1 DESC
""").fetchall()
for r in conf_rows:
    print(f'  {r[0]}: {r[1]}')

print('\n=== BRIER REWARD PREVIEW (sample) ===')
sample = conn.execute("""
    SELECT jv.v_label, ao.agent_role, ao.round_num,
           ao.confidence_internal,
           ROUND(2*ao.confidence_internal*jv.v_label - ao.confidence_internal*ao.confidence_internal, 4) as brier
    FROM judge_verdicts jv
    JOIN agent_outputs ao USING (claim_id)
    WHERE jv.v_label IS NOT NULL AND ao.round_num=1
    LIMIT 9
""").fetchall()
print(f'  {"v_label":<8} {"agent":<8} {"R":<3} {"conf":<6} {"brier":<8}')
for r in sample:
    print(f'  {r[0]:<8} {r[1]:<8} {r[2]:<3} {r[3]:<6.2f} {r[4]:<8}')

conn.close()


=== TABLE COUNTS ===
  queries              50
  claims               414
  agent_outputs        2484
  agent_deltas         1242
  judge_verdicts       414

=== JUDGE VERDICT DISTRIBUTION ===
  NOT_SUPPORTED (0.0)     256 (61%) | avg_conf=0.888
  PARTIAL (0.5)            95 (22%) | avg_conf=0.789
  SUPPORTED (1.0)          63 (15%) | avg_conf=0.902

=== MISSING JUDGE VERDICTS ===
  Claims without verdict: 0

=== JUDGE vs AGENT AGREEMENT (R1) ===
  Judge agrees with agent majority: 330/414 (79%)
  ~ Moderate agreement (79%) — judge partially independent

=== JUDGE CONFIDENCE DISTRIBUTION ===
  0.90-0.95: 250
  0.80-0.89: 138
  0.70-0.79: 26

=== BRIER REWARD PREVIEW (sample) ===
  v_label  agent    R   conf   brier   
  0.5      agent_a  1   0.85   0.1275  
  1.0      agent_a  1   0.90   0.99    
  1.0      agent_b  1   0.90   0.99    
  1.0      agent_c  1   0.95   0.9975  
  0.5      agent_b  1   0.85   0.1275  
  0.5      agent_c  1   0.90   0.09    
  0.5      agent_a  1   0.90   0

In [ ]:
# CELL 14 — Re-export GRPO / SFT / DPO with corrected Brier rewards
# BUG FIX: confidence_internal = "certainty in own verdict", NOT P(claim is true).
# For NOT_SUPPORTED verdicts: p_for_brier = 1 - confidence_internal
# Formula: symmetric Brier R = 1 - 2*(p - v_label)^2  → range [-1, +1]
#   Perfect correct: R = +1.0  |  Total random: R = 0.5  |  Perfect wrong: R = -1.0
import json, sqlite3
from collections import defaultdict

MAX_CHUNK_CHARS = 800

SYSTEM_PROMPTS = {
    'agent_a': (
        'You are a strict regulatory compliance verifier in a multi-agent debate.\n'
        'YOUR ROLE: Find the strongest case FOR the claim being correct, based ONLY on the retrieved evidence.'
    ),
    'agent_b': (
        'You are an adversarial auditor in a multi-agent regulatory compliance debate.\n'
        'YOUR ROLE: Find what is WRONG, UNSUPPORTED, or MISLEADING about the claim using ONLY the retrieved evidence.'
    ),
    'agent_c': (
        'You are a neutral evidence calibrator in a multi-agent regulatory compliance debate.\n'
        'YOUR ROLE: Weigh the evidence honestly on BOTH sides. You have NO prior stance.'
    ),
}

def p_for_brier(confidence: float, verdict: str) -> float:
    """Convert agent confidence_internal to P(claim is TRUE) for Brier scoring.
    confidence_internal = certainty in OWN verdict, not P(claim is true).
    NOT_SUPPORTED with conf=0.85 means P(true) = 1-0.85 = 0.15.
    PARTIAL maps to 0.5 regardless of confidence (partial support = 50%).
    """
    if verdict == 'NOT_SUPPORTED':
        return 1.0 - confidence
    elif verdict == 'PARTIAL':
        return 0.5
    elif verdict == 'SUPPORTED':
        return confidence
    else:  # IDK
        return 0.5

def brier_reward(confidence: float, verdict: str, v_label: float) -> float:
    """Symmetric Brier reward: R = 1 - 2*(p - v_label)^2
    Range: [-1, +1].  Perfect correct = +1.0.  Perfect wrong = -1.0.  Random = +0.5.
    Correctly rewards both SUPPORTED and NOT_SUPPORTED when agent is right.
    """
    p = p_for_brier(confidence, verdict)
    return round(1.0 - 2.0 * (p - v_label) ** 2, 4)

def export_for_unsloth(db_path: str, out_prefix: str = '/content/mad_v2'):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    rows = conn.execute("""
        SELECT
            ao.output_id, ao.claim_id, ao.agent_role, ao.round_num,
            ao.verdict, ao.reasoning, ao.confidence_internal, ao.raw_response,
            ao.parse_failed, ao.call_failed, ao.latency_ms,
            c.claim_text, c.is_material, c.is_critical, c.confidence_prior,
            q.user_query, q.rag_chunks, q.baseline_answer, q.query_id,
            jv.v_label, jv.judge_confidence, jv.judge_reasoning
        FROM agent_outputs ao
        JOIN claims  c  USING (claim_id)
        JOIN queries q  USING (query_id)
        LEFT JOIN judge_verdicts jv USING (claim_id)
        ORDER BY q.query_id, c.claim_index, ao.agent_role, ao.round_num
    """).fetchall()
    conn.close()

    grpo_records, sft_records, dpo_pairs = [], [], []
    by_claim = defaultdict(list)
    for r in rows:
        by_claim[r['claim_id']].append(dict(r))

    for claim_id, claim_rows in by_claim.items():
        chunks_raw  = json.loads(claim_rows[0]['rag_chunks'])
        chunks_text = '\n\n'.join(
            f"[{c['chunk_id']}] {c['text'][:MAX_CHUNK_CHARS]}" for c in chunks_raw
        )

        for r in claim_rows:
            v_label = r['v_label']
            reward  = (brier_reward(r['confidence_internal'], r['verdict'], v_label)
                       if v_label is not None else None)

            up = (
                f"USER QUERY: {r['user_query']}\n\n"
                f"CLAIM TO VERIFY:\n{r['claim_text']}\n\n"
                f"RETRIEVED EVIDENCE:\n{chunks_text}\n\n"
                f"Provide your verdict."
            )

            grpo_records.append({
                'id':            r['output_id'],
                'query_id':      r['query_id'],
                'claim_id':      claim_id,
                'agent_role':    r['agent_role'],
                'round_num':     r['round_num'],
                'is_material':   bool(r['is_material']),
                'is_critical':   bool(r['is_critical']),
                'system':        SYSTEM_PROMPTS.get(r['agent_role'], ''),
                'prompt':        up,
                'completion':    r['raw_response'] or '',
                'verdict':       r['verdict'],
                'confidence':    r['confidence_internal'],
                'p_for_brier':   p_for_brier(r['confidence_internal'], r['verdict']),
                'v_label':       v_label,
                'brier_reward':  reward,
                'parse_failed':  bool(r['parse_failed']),
                'call_failed':   bool(r['call_failed']),
                'latency_ms':    r['latency_ms'],
                'judge_reasoning': r['judge_reasoning'],
            })

            # SFT: R1 outputs where agent verdict was correct and Brier > 0.5
            # With symmetric formula, this captures good NOT_SUPPORTED calls too
            if (not r['parse_failed'] and not r['call_failed']
                    and reward is not None and reward > 0.5
                    and r['round_num'] == 1):
                sft_records.append({
                    'system':     SYSTEM_PROMPTS.get(r['agent_role'], ''),
                    'prompt':     up,
                    'completion': r['raw_response'] or '',
                    'metadata': {
                        'claim_id':   claim_id,
                        'agent_role': r['agent_role'],
                        'verdict':    r['verdict'],
                        'confidence': r['confidence_internal'],
                        'p_brier':    p_for_brier(r['confidence_internal'], r['verdict']),
                        'brier':      reward,
                        'v_label':    v_label,
                    }
                })

        # DPO: winner vs loser per claim based on Brier reward (R1 only, after judge)
        r1s = [r for r in claim_rows
               if r['round_num'] == 1 and r['v_label'] is not None
               and not r['parse_failed'] and not r['call_failed']]

        if len(r1s) >= 2:
            scored = sorted(
                r1s,
                key=lambda r: brier_reward(r['confidence_internal'], r['verdict'], r['v_label']),
                reverse=True
            )
            winner, loser = scored[0], scored[-1]
            w_r = brier_reward(winner['confidence_internal'], winner['verdict'], winner['v_label'])
            l_r = brier_reward(loser['confidence_internal'],  loser['verdict'],  loser['v_label'])
            if w_r > l_r + 0.15:  # meaningful margin
                dpo_pairs.append({
                    'prompt':   (
                        f"USER QUERY: {winner['user_query']}\n\n"
                        f"CLAIM TO VERIFY:\n{winner['claim_text']}\n\n"
                        f"RETRIEVED EVIDENCE:\n{chunks_text}"
                    ),
                    'chosen':   winner['raw_response'] or '',
                    'rejected': loser['raw_response']  or '',
                    'metadata': {
                        'claim_id':      claim_id,
                        'winner_agent':  winner['agent_role'],
                        'loser_agent':   loser['agent_role'],
                        'winner_verdict': winner['verdict'],
                        'loser_verdict':  loser['verdict'],
                        'winner_reward': w_r,
                        'loser_reward':  l_r,
                        'v_label':       winner['v_label'],
                    }
                })

    paths = {}
    for name, data in [('grpo', grpo_records), ('sft', sft_records), ('dpo', dpo_pairs)]:
        path = f'{out_prefix}_{name}.jsonl'
        with open(path, 'w') as f:
            for rec in data:
                f.write(json.dumps(rec) + '\n')
        paths[name] = path
        print(f'  {name:<6} {len(data):>5} records  ->  {path}')

    # Reward stats
    rewards = [r['brier_reward'] for r in grpo_records if r['brier_reward'] is not None]
    if rewards:
        print(f'\n  Symmetric Brier reward stats:')
        print(f'    min={min(rewards):.3f}  avg={sum(rewards)/len(rewards):.3f}  max={max(rewards):.3f}')
        pos = sum(1 for r in rewards if r > 0.5)
        neg = sum(1 for r in rewards if r < 0)
        print(f'    reward>0.5 (correct+confident): {pos}/{len(rewards)} ({pos*100//len(rewards)}%)')
        print(f'    reward<0   (wrong calls):        {neg}/{len(rewards)} ({neg*100//len(rewards)}%)')

    # Per-verdict breakdown
    from collections import defaultdict as dd
    by_v = dd(list)
    for r in grpo_records:
        if r['brier_reward'] is not None:
            by_v[r['verdict']].append(r['brier_reward'])
    print(f'\n  Avg Brier by verdict (all rounds):')
    for v, rs in sorted(by_v.items()):
        print(f'    {v:<15}: avg={sum(rs)/len(rs):+.3f}  n={len(rs)}')

    return paths

print('✓ Running export with fixed Brier reward...')
paths = export_for_unsloth(DB_PATH)


✓ Running export with fixed Brier reward...
  grpo    2484 records  ->  /content/mad_v2_grpo.jsonl
  sft     1221 records  ->  /content/mad_v2_sft.jsonl
  dpo       77 records  ->  /content/mad_v2_dpo.jsonl

  Symmetric Brier reward stats:
    min=-0.805  avg=0.865  max=1.000
    reward>0.5 (correct+confident): 2258/2484 (90%)
    reward<0   (wrong calls):        65/2484 (2%)

  Avg Brier by verdict (all rounds):
    IDK            : avg=+0.500  n=15
    NOT_SUPPORTED  : avg=+0.922  n=1528
    PARTIAL        : avg=+0.801  n=336
    SUPPORTED      : avg=+0.764  n=605


In [ ]:
# # CELL 14 — Re-export GRPO / SFT / DPO with Brier rewards
# # Now that judge_verdicts is filled, v_label and brier_reward will be populated.
# import json, sqlite3
# from collections import defaultdict

# # Constants from Stage 1 (must match)
# MAX_CHUNK_CHARS = 800
# CONF_MIN, CONF_MAX = 0.05, 0.95

# SYSTEM_PROMPTS = {
#     'agent_a': (
#         'You are a strict regulatory compliance verifier in a multi-agent debate.\n'
#         'YOUR ROLE: Find the strongest case FOR the claim being correct, based ONLY on the retrieved evidence.'
#     ),
#     'agent_b': (
#         'You are an adversarial auditor in a multi-agent regulatory compliance debate.\n'
#         'YOUR ROLE: Find what is WRONG, UNSUPPORTED, or MISLEADING about the claim using ONLY the retrieved evidence.'
#     ),
#     'agent_c': (
#         'You are a neutral evidence calibrator in a multi-agent regulatory compliance debate.\n'
#         'YOUR ROLE: Weigh the evidence honestly on BOTH sides. You have NO prior stance.'
#     ),
# }

# def brier_reward(confidence: float, v_label: float) -> float:
#     """R = 2*p*v - p^2. Range [-1, 1]. Maximized when p=v."""
#     return round(2 * confidence * v_label - confidence ** 2, 4)

# def export_for_unsloth(db_path: str, out_prefix: str = '/content/mad_v2'):
#     conn = sqlite3.connect(db_path)
#     conn.row_factory = sqlite3.Row

#     rows = conn.execute("""
#         SELECT
#             ao.output_id, ao.claim_id, ao.agent_role, ao.round_num,
#             ao.verdict, ao.reasoning, ao.confidence_internal, ao.raw_response,
#             ao.parse_failed, ao.call_failed, ao.latency_ms,
#             c.claim_text, c.is_material, c.is_critical, c.confidence_prior,
#             q.user_query, q.rag_chunks, q.baseline_answer, q.query_id,
#             jv.v_label, jv.judge_confidence, jv.judge_reasoning
#         FROM agent_outputs ao
#         JOIN claims  c  USING (claim_id)
#         JOIN queries q  USING (query_id)
#         LEFT JOIN judge_verdicts jv USING (claim_id)
#         ORDER BY q.query_id, c.claim_index, ao.agent_role, ao.round_num
#     """).fetchall()
#     conn.close()

#     grpo_records, sft_records, dpo_pairs = [], [], []
#     by_claim = defaultdict(list)
#     for r in rows:
#         by_claim[r['claim_id']].append(dict(r))

#     for claim_id, claim_rows in by_claim.items():
#         chunks_raw  = json.loads(claim_rows[0]['rag_chunks'])
#         chunks_text = '\n\n'.join(
#             f"[{c['chunk_id']}] {c['text'][:MAX_CHUNK_CHARS]}" for c in chunks_raw
#         )

#         for r in claim_rows:
#             v_label = r['v_label']
#             reward  = brier_reward(r['confidence_internal'], v_label) if v_label is not None else None

#             up = (
#                 f"USER QUERY: {r['user_query']}\n\n"
#                 f"CLAIM TO VERIFY:\n{r['claim_text']}\n\n"
#                 f"RETRIEVED EVIDENCE:\n{chunks_text}\n\n"
#                 f"Provide your verdict."
#             )

#             grpo_records.append({
#                 'id':           r['output_id'],
#                 'query_id':     r['query_id'],
#                 'claim_id':     claim_id,
#                 'agent_role':   r['agent_role'],
#                 'round_num':    r['round_num'],
#                 'is_material':  bool(r['is_material']),
#                 'is_critical':  bool(r['is_critical']),
#                 'system':       SYSTEM_PROMPTS.get(r['agent_role'], ''),
#                 'prompt':       up,
#                 'completion':   r['raw_response'] or '',
#                 'verdict':      r['verdict'],
#                 'confidence':   r['confidence_internal'],
#                 'v_label':      v_label,
#                 'brier_reward': reward,
#                 'parse_failed': bool(r['parse_failed']),
#                 'call_failed':  bool(r['call_failed']),
#                 'latency_ms':   r['latency_ms'],
#                 'judge_reasoning': r['judge_reasoning'],
#             })

#             # SFT: clean R1 outputs where judge confirmed the agent was correct
#             if (not r['parse_failed'] and not r['call_failed']
#                     and reward is not None and reward > 0.5
#                     and r['round_num'] == 1):
#                 sft_records.append({
#                     'system':     SYSTEM_PROMPTS.get(r['agent_role'], ''),
#                     'prompt':     up,
#                     'completion': r['raw_response'] or '',
#                     'metadata': {
#                         'claim_id': claim_id, 'agent_role': r['agent_role'],
#                         'verdict': r['verdict'], 'confidence': r['confidence_internal'],
#                         'brier': reward, 'v_label': v_label,
#                     }
#                 })

#         # DPO: best vs worst agent on same claim (R1, after judge label)
#         r1s = [r for r in claim_rows
#                if r['round_num'] == 1 and r['v_label'] is not None
#                and not r['parse_failed'] and not r['call_failed']]

#         if len(r1s) >= 2:
#             scored = sorted(r1s,
#                 key=lambda r: brier_reward(r['confidence_internal'], r['v_label']),
#                 reverse=True)
#             winner, loser = scored[0], scored[-1]
#             w_r = brier_reward(winner['confidence_internal'], winner['v_label'])
#             l_r = brier_reward(loser['confidence_internal'],  loser['v_label'])
#             if w_r > l_r + 0.1:
#                 dpo_pairs.append({
#                     'prompt':   (
#                         f"USER QUERY: {winner['user_query']}\n\n"
#                         f"CLAIM TO VERIFY:\n{winner['claim_text']}\n\n"
#                         f"RETRIEVED EVIDENCE:\n{chunks_text}"
#                     ),
#                     'chosen':   winner['raw_response'] or '',
#                     'rejected': loser['raw_response']  or '',
#                     'metadata': {
#                         'claim_id': claim_id,
#                         'winner_agent': winner['agent_role'], 'loser_agent': loser['agent_role'],
#                         'winner_reward': w_r, 'loser_reward': l_r, 'v_label': winner['v_label'],
#                     }
#                 })

#     paths = {}
#     for name, data in [('grpo', grpo_records), ('sft', sft_records), ('dpo', dpo_pairs)]:
#         path = f'{out_prefix}_{name}.jsonl'
#         with open(path, 'w') as f:
#             for rec in data:
#                 f.write(json.dumps(rec) + '\n')
#         paths[name] = path
#         print(f'  {name:<6} {len(data):>5} records  ->  {path}')

#     # Quick reward stats
#     rewards = [r['brier_reward'] for r in grpo_records if r['brier_reward'] is not None]
#     if rewards:
#         print(f'\n  Brier reward: min={min(rewards):.3f} avg={sum(rewards)/len(rewards):.3f} max={max(rewards):.3f}')
#         good = sum(1 for r in rewards if r > 0.5)
#         print(f'  High-quality outputs (reward>0.5): {good}/{len(rewards)} ({good*100//len(rewards)}%)')
#     return paths

# print('\u2713 Running export...')
# paths = export_for_unsloth(DB_PATH)

✓ Running export...
  grpo    2484 records  ->  /content/mad_v2_grpo.jsonl
  sft      189 records  ->  /content/mad_v2_sft.jsonl
  dpo       60 records  ->  /content/mad_v2_dpo.jsonl

  Brier reward: min=-0.902 avg=-0.281 max=0.998
  High-quality outputs (reward>0.5): 378/2484 (15%)


In [ ]:
# CELL 15 — Download DB + all export files
from google.colab import files
import os

print('Downloading...')

# DB with judge_verdicts filled
print(f'  DB: {DB_PATH}')
files.download(DB_PATH)

# GRPO exports
for name in ['grpo', 'sft', 'dpo']:
    path = f'/content/mad_v2_{name}.jsonl'
    if os.path.exists(path) and os.path.getsize(path) > 0:
        kb = os.path.getsize(path) / 1024
        print(f'  {name}.jsonl: {kb:.1f} KB')
        files.download(path)
    else:
        print(f'  {name}.jsonl: empty or missing — run Cell 14 first')

print('\nDone.')

Downloading...
  DB: /content/mad_before_phase1_5090_ragfix_01_ (2) (1) (1).db


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  grpo.jsonl: 13478.7 KB


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  dpo.jsonl: 462.8 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done.


## What This Notebook Produces

| File | Contents | Used For |
|---|---|---|
| `mad_v2_grpo.jsonl` | All 2484 outputs with `brier_reward` filled | GRPO training reward signal |
| `mad_v2_sft.jsonl` | High-quality R1 outputs (reward > 0.5) | Supervised fine-tuning warm-start |
| `mad_v2_dpo.jsonl` | Winner/loser pairs per claim | DPO preference training |
| Updated `.db` | `judge_verdicts` filled | Source of truth for all exports |

## Bias Controls Applied

| Control | How |
|---|---|
| Agent identity hidden | Judge sees `Debater 1/2/3`, not `agent_a/b/c` |
| Confidence hidden | `confidence_internal` never shown to judge |
| Agent order randomized | Per-claim shuffle seeded by `claim_id` (reproducible) |
| Anti-majority instruction | Explicit: "consensus is NOT evidence" |
| Evidence-first protocol | Judge forms own verdict before reading debate |
| Deterministic judge | `temperature=0.0` — consistent labels |

## Interview Story
```
14B AWQ agents (stochastic, temp 0.3/0.8/0.5) generate debate signal.
Same 14B AWQ model as judge (deterministic, temp 0.0) produces final labels.
Prefix caching: judge system prompt is identical for all 414 claims → high cache hit rate.
Agent calls were intentionally stochastic (no caching). Judge calls are deterministic.
```